In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [30]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision import transforms

In [31]:
import torch

In [ ]:
#I'll be using data that I found on the pytorch website - its an image dataset called Flowers102

In [46]:
#Loading data

transform = transforms.Compose([
    transforms.Resize((28, 28)),  # Resize images to 224x224 (or any fixed size)
    transforms.ToTensor(),
])

training_data = datasets.Flowers102(
    root="data",
    split="train",
    download=True,
    transform=transform,
)

test_data = datasets.Flowers102(
    root="data",
    split="train",
    download=True,
    transform=transform,
)

In [47]:
batch_size = 64

#data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [70]:
#defining the model 
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(2352, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 102)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [81]:
#loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-1)

In [82]:
print(f"Dataset size: {len(train_dataloader)}")

Dataset size: 16


In [83]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        y_pred = model(X)
        print(y.min(), y.max())
        
        loss = loss_fn(y_pred, y)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 1 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [84]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [85]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
tensor(0) tensor(6)
loss: 4.635611  [   64/ 1020]
tensor(6) tensor(12)
loss: 4.594641  [  128/ 1020]
tensor(12) tensor(20)
loss: 4.670448  [  192/ 1020]
tensor(18) tensor(25)
loss: 4.666456  [  256/ 1020]
tensor(25) tensor(31)
loss: 4.633562  [  320/ 1020]
tensor(32) tensor(38)
loss: 4.666390  [  384/ 1020]
tensor(38) tensor(44)
loss: 4.681258  [  448/ 1020]
tensor(44) tensor(51)
loss: 4.694285  [  512/ 1020]
tensor(51) tensor(57)
loss: 4.679727  [  576/ 1020]
tensor(57) tensor(63)
loss: 4.666063  [  640/ 1020]
tensor(64) tensor(70)
loss: 4.687841  [  704/ 1020]
tensor(70) tensor(76)
loss: 4.672024  [  768/ 1020]
tensor(76) tensor(83)
loss: 4.689148  [  832/ 1020]
tensor(83) tensor(89)
loss: 4.675621  [  896/ 1020]
tensor(89) tensor(95)
loss: 4.677847  [  960/ 1020]
tensor(96) tensor(101)
loss: 4.659217  [  960/ 1020]
Test Error: 
 Accuracy: 1.3%, Avg loss: 4.619881 

Epoch 2
-------------------------------
tensor(0) tensor(6)
loss: 4.616261  [  

In [ ]:
#The model is currently very innacurate when it comes to this dataset, It would probably need many more epochs to become more accurate

In [ ]:
#However the model is steadily improving with each epoch